In [325]:
# Chapter 11
## SNPBLUP
## GBLUP
## Computing SNP solutions from GBLUP
## Computing base population allele frequencies

In [326]:
using DataFrames
using StatsBase
using LinearAlgebra

In [327]:
# Example 11.2 ------------------------------------------------------------

In [328]:
# Animals 21 to 26 are assumed as the selection candidates
a = collect(13:26)
s = [missing, missing, 13, 15, 15, 14, 14, 14, 1, 14, 14, 14, 14, 14]
d = [missing, missing, 4, 2, 5, 6, 9, 9, 3, 8, 11, 10, 7, 12]
dyd = [9.0, 13.4, 12.7, 15.4, 5.9, 7.7, 10.2, 4.8, missing, missing, missing, missing, missing, missing]  

14-element Vector{Union{Missing, Float64}}:
  9.0
 13.4
 12.7
 15.4
  5.9
  7.7
 10.2
  4.8
   missing
   missing
   missing
   missing
   missing
   missing

In [329]:
data = DataFrame(a=a,s=s,d=d,dyd=dyd)

Row,a,s,d,dyd
,Int64,Int64?,Int64?,Float64?
1,13,missing,missing,9.0
2,14,missing,missing,13.4
3,15,13,4,12.7
4,16,15,2,15.4
5,17,15,5,5.9
6,18,14,6,7.7
7,19,14,9,10.2
8,20,14,9,4.8
9,21,1,3,missing


In [330]:
geno = [
    2 0 1 1 0 0 0 2 1 2;
    1 0 0 0 0 2 0 2 1 0;
    1 1 2 1 1 0 0 2 1 2;
    0 0 2 1 0 1 0 2 2 1;
    0 1 1 2 0 0 0 2 1 2;
    1 1 0 1 0 2 0 2 2 1;
    0 0 1 1 0 2 0 2 2 0;
    0 1 1 0 0 1 0 2 2 0;
    2 0 0 0 0 1 2 2 1 2;
    0 0 0 1 1 2 0 2 0 0;
    0 1 1 0 0 1 0 2 2 1;
    1 0 0 0 1 1 0 2 0 0;
    0 0 0 1 1 2 0 2 1 0;
    1 0 1 1 0 2 0 1 0 0
]

14×10 Matrix{Int64}:
 2  0  1  1  0  0  0  2  1  2
 1  0  0  0  0  2  0  2  1  0
 1  1  2  1  1  0  0  2  1  2
 0  0  2  1  0  1  0  2  2  1
 0  1  1  2  0  0  0  2  1  2
 1  1  0  1  0  2  0  2  2  1
 0  0  1  1  0  2  0  2  2  0
 0  1  1  0  0  1  0  2  2  0
 2  0  0  0  0  1  2  2  1  2
 0  0  0  1  1  2  0  2  0  0
 0  1  1  0  0  1  0  2  2  1
 1  0  0  0  1  1  0  2  0  0
 0  0  0  1  1  2  0  2  1  0
 1  0  1  1  0  2  0  1  0  0

In [331]:
geno = Matrix{Union{Float64,Int64}}(geno)

14×10 Matrix{Union{Float64, Int64}}:
 2  0  1  1  0  0  0  2  1  2
 1  0  0  0  0  2  0  2  1  0
 1  1  2  1  1  0  0  2  1  2
 0  0  2  1  0  1  0  2  2  1
 0  1  1  2  0  0  0  2  1  2
 1  1  0  1  0  2  0  2  2  1
 0  0  1  1  0  2  0  2  2  0
 0  1  1  0  0  1  0  2  2  0
 2  0  0  0  0  1  2  2  1  2
 0  0  0  1  1  2  0  2  0  0
 0  1  1  0  0  1  0  2  2  1
 1  0  0  0  1  1  0  2  0  0
 0  0  0  1  1  2  0  2  1  0
 1  0  1  1  0  2  0  1  0  0

In [332]:
p = mean(geno,dims=1) ./ 2

1×10 Matrix{Float64}:
 0.321429  0.178571  0.357143  0.357143  …  0.964286  0.571429  0.392857

In [333]:
#center genotypes
geno .-= 2 .*p

14×10 Matrix{Union{Float64, Int64}}:
  1.35714   -0.357143   0.285714  …   0.0714286  -0.142857   1.21429
  0.357143  -0.357143  -0.714286      0.0714286  -0.142857  -0.785714
  0.357143   0.642857   1.28571       0.0714286  -0.142857   1.21429
 -0.642857  -0.357143   1.28571       0.0714286   0.857143   0.214286
 -0.642857   0.642857   0.285714      0.0714286  -0.142857   1.21429
  0.357143   0.642857  -0.714286  …   0.0714286   0.857143   0.214286
 -0.642857  -0.357143   0.285714      0.0714286   0.857143  -0.785714
 -0.642857   0.642857   0.285714      0.0714286   0.857143  -0.785714
  1.35714   -0.357143  -0.714286      0.0714286  -0.142857   1.21429
 -0.642857  -0.357143  -0.714286      0.0714286  -1.14286   -0.785714
 -0.642857   0.642857   0.285714  …   0.0714286   0.857143   0.214286
  0.357143  -0.357143  -0.714286      0.0714286  -1.14286   -0.785714
 -0.642857  -0.357143  -0.714286      0.0714286  -0.142857  -0.785714
  0.357143  -0.357143   0.285714     -0.928571   -1.14286

In [334]:
geno = hcat(a,geno)

14×11 Matrix{Union{Float64, Int64}}:
 13   1.35714   -0.357143   0.285714  …   0.0714286  -0.142857   1.21429
 14   0.357143  -0.357143  -0.714286      0.0714286  -0.142857  -0.785714
 15   0.357143   0.642857   1.28571       0.0714286  -0.142857   1.21429
 16  -0.642857  -0.357143   1.28571       0.0714286   0.857143   0.214286
 17  -0.642857   0.642857   0.285714      0.0714286  -0.142857   1.21429
 18   0.357143   0.642857  -0.714286  …   0.0714286   0.857143   0.214286
 19  -0.642857  -0.357143   0.285714      0.0714286   0.857143  -0.785714
 20  -0.642857   0.642857   0.285714      0.0714286   0.857143  -0.785714
 21   1.35714   -0.357143  -0.714286      0.0714286  -0.142857   1.21429
 22  -0.642857  -0.357143  -0.714286      0.0714286  -1.14286   -0.785714
 23  -0.642857   0.642857   0.285714  …   0.0714286   0.857143   0.214286
 24   0.357143  -0.357143  -0.714286      0.0714286  -1.14286   -0.785714
 25  -0.642857  -0.357143  -0.714286      0.0714286  -0.142857  -0.785714
 26  

In [335]:
genoRef = geno[1:8,2:end]

8×10 Matrix{Union{Float64, Int64}}:
  1.35714   -0.357143   0.285714  …  0.0714286  -0.142857   1.21429
  0.357143  -0.357143  -0.714286     0.0714286  -0.142857  -0.785714
  0.357143   0.642857   1.28571      0.0714286  -0.142857   1.21429
 -0.642857  -0.357143   1.28571      0.0714286   0.857143   0.214286
 -0.642857   0.642857   0.285714     0.0714286  -0.142857   1.21429
  0.357143   0.642857  -0.714286  …  0.0714286   0.857143   0.214286
 -0.642857  -0.357143   0.285714     0.0714286   0.857143  -0.785714
 -0.642857   0.642857   0.285714     0.0714286   0.857143  -0.785714

In [336]:
genoCand = geno[9:14,2:end]

6×10 Matrix{Union{Float64, Int64}}:
  1.35714   -0.357143  -0.714286  …   0.0714286  -0.142857   1.21429
 -0.642857  -0.357143  -0.714286      0.0714286  -1.14286   -0.785714
 -0.642857   0.642857   0.285714      0.0714286   0.857143   0.214286
  0.357143  -0.357143  -0.714286      0.0714286  -1.14286   -0.785714
 -0.642857  -0.357143  -0.714286      0.0714286  -0.142857  -0.785714
  0.357143  -0.357143   0.285714  …  -0.928571   -1.14286   -0.785714

In [337]:
# Variance ratios 
varA = 35.241 
varE = 245
k = 2 * (sum(p .* (1.0 .- p)))
αG = varE / varA
αM = k*(varE / varA)

24.598479044295

In [338]:
# Setting up the incidence matrices for the MME

In [339]:
#rows with data
dataRows = .!ismissing.(data.dyd)

14-element BitVector:
 1
 1
 1
 1
 1
 1
 1
 1
 0
 0
 0
 0
 0
 0

In [340]:
X = ones(sum(dataRows))

8-element Vector{Float64}:
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0
 1.0

In [341]:
y = data.dyd[dataRows]

8-element Vector{Union{Missing, Float64}}:
  9.0
 13.4
 12.7
 15.4
  5.9
  7.7
 10.2
  4.8

In [342]:
# Components of SNPBLUP MME

In [343]:
X'X

8.0

In [344]:
X'genoRef

1×10 adjoint(::Vector{Float64}) with eltype Float64:
 -0.142857  1.14286  2.28571  1.28571  …  0.571429  2.85714  1.71429

In [345]:
genoRef'X

10-element Vector{Float64}:
 -0.1428571428571428
  1.1428571428571426
  2.2857142857142847
  1.2857142857142851
 -1.2857142857142856
 -1.714285714285714
 -1.1428571428571426
  0.5714285714285712
  2.8571428571428577
  1.7142857142857153

In [346]:
genoRef'genoRef+Matrix(1.0I,size(genoRef,2),size(genoRef,2))*αM

10×10 Matrix{Float64}:
 28.476      -0.520408   -1.04082   …  -0.0102041  -1.55102     1.96939
 -0.520408   26.7617      0.326531      0.0816327   0.408163    1.2449
 -1.04082     0.326531   29.2515        0.163265    0.816327    2.4898
 -0.397959    0.683673    1.36735       0.0918367  -0.0408163   3.27551
  0.397959    0.316327    0.632653     -0.0918367  -0.959184    0.72449
 -0.969388   -1.2449     -3.4898    …  -0.122449    1.38776    -5.36735
  0.0204082  -0.163265   -0.326531     -0.0816327  -0.408163   -0.244898
 -0.0102041   0.0816327   0.163265     24.6393      0.204082    0.122449
 -1.55102     0.408163    0.816327      0.204082   27.6189     -1.38776
  1.96939     1.2449      2.4898        0.122449   -1.38776    30.9658

In [347]:
# MME

In [348]:
LHS = [X'X X'genoRef;
       genoRef'X genoRef'genoRef+Matrix(1.0I,size(genoRef,2),size(genoRef,2))*αM]

11×11 Matrix{Float64}:
  8.0       -0.142857    1.14286    …   0.571429    2.85714     1.71429
 -0.142857  28.476      -0.520408      -0.0102041  -1.55102     1.96939
  1.14286   -0.520408   26.7617         0.0816327   0.408163    1.2449
  2.28571   -1.04082     0.326531       0.163265    0.816327    2.4898
  1.28571   -0.397959    0.683673       0.0918367  -0.0408163   3.27551
 -1.28571    0.397959    0.316327   …  -0.0918367  -0.959184    0.72449
 -1.71429   -0.969388   -1.2449        -0.122449    1.38776    -5.36735
 -1.14286    0.0204082  -0.163265      -0.0816327  -0.408163   -0.244898
  0.571429  -0.0102041   0.0816327     24.6393      0.204082    0.122449
  2.85714   -1.55102     0.408163       0.204082   27.6189     -1.38776
  1.71429    1.96939     1.2449     …   0.122449   -1.38776    30.9658

In [349]:
RHS = [X'y;
       genoRef'y]

11-element Vector{Union{Missing, Float64}}:
  79.1
   0.9499999999999988
   2.8499999999999974
  29.599999999999994
  10.299999999999999
  -9.9
 -13.249999999999991
 -11.299999999999999
   5.649999999999998
  26.800000000000004
  16.15

In [350]:
sol = LHS\RHS

11-element Vector{Float64}:
  9.94394205161951
  0.08702066029092966
 -0.3107911916704227
  0.2624591949550949
 -0.08027688756497668
  0.11020776768703461
  0.1390797845002317
 -7.11882656703362e-17
  3.5818829372783896e-17
 -0.06069022928156946
 -0.015802326618122385

In [351]:
# Compute DGV for the reference population 
uHatRef = genoRef*sol[2:end]

8-element Vector{Float64}:
  0.0702593737316594
  0.11082062828731967
  0.04511448441243651
  0.25286912994167954
 -0.4948500260855993
 -0.3567400068477715
  0.14529204610493876
 -0.2243020425007389

In [352]:
# Compute DGV for the selection candidates
uHatCand = genoCand*sol[2:end]

6-element Vector{Float64}:
  0.027156850841772714
  0.11442107740001739
 -0.2401043691188613
  0.14263884075569203
  0.05373084811844793
  0.3536931649590072

In [353]:
# Re-analyse using EDCs as weights
edc = [558, 722, 300, 73, 52, 87, 64, 103, missing, missing, missing, missing, missing, missing]

14-element Vector{Union{Missing, Int64}}:
 558
 722
 300
  73
  52
  87
  64
 103
    missing
    missing
    missing
    missing
    missing
    missing

In [354]:
data.edc = edc
data

Row,a,s,d,dyd,edc
,Int64,Int64?,Int64?,Float64?,Int64?
1,13,missing,missing,9.0,558
2,14,missing,missing,13.4,722
3,15,13,4,12.7,300
4,16,15,2,15.4,73
5,17,15,5,5.9,52
6,18,14,6,7.7,87
7,19,14,9,10.2,64
8,20,14,9,4.8,103
9,21,1,3,missing,missing


In [355]:
dii = (1.0 ./edc[dataRows])
D = diagm(dii)

8×8 Matrix{Float64}:
 0.00179211  0.0         0.0         …  0.0        0.0       0.0
 0.0         0.00138504  0.0            0.0        0.0       0.0
 0.0         0.0         0.00333333     0.0        0.0       0.0
 0.0         0.0         0.0            0.0        0.0       0.0
 0.0         0.0         0.0            0.0        0.0       0.0
 0.0         0.0         0.0         …  0.0114943  0.0       0.0
 0.0         0.0         0.0            0.0        0.015625  0.0
 0.0         0.0         0.0            0.0        0.0       0.00970874

In [356]:
Dinv = diagm(edc[dataRows])

8×8 Matrix{Union{Missing, Int64}}:
 558    0    0   0   0   0   0    0
   0  722    0   0   0   0   0    0
   0    0  300   0   0   0   0    0
   0    0    0  73   0   0   0    0
   0    0    0   0  52   0   0    0
   0    0    0   0   0  87   0    0
   0    0    0   0   0   0  64    0
   0    0    0   0   0   0   0  103

In [357]:
# Components of MME

In [358]:
X'*Dinv*X

1959.0

In [359]:
X'*Dinv*genoRef

1×10 adjoint(::Vector{Union{Missing, Float64}}) with eltype Union{Missing, Float64}:
 965.643  -157.643  123.714  -213.286  …  -279.857  139.929  47.1429  440.786

In [360]:
genoRef'*Dinv*X

10-element Vector{Union{Missing, Float64}}:
  965.6428571428573
 -157.64285714285717
  123.7142857142856
 -213.28571428571433
 -259.71428571428567
 -456.78571428571416
 -279.85714285714283
  139.92857142857136
   47.142857142857295
  440.78571428571445

In [361]:
genoRef'*Dinv*genoRef

10×10 Matrix{Union{Missing, Float64, Int64}}:
 1289.87    -306.301      47.1837   …   68.9745   -261.163      887.423
 -306.301    404.73      323.673       -11.2602     95.7347     207.719
   47.1837   323.673    1092.78          8.83673    61.7551    1054.51
   50.8265   180.031     536.49        -15.2347     20.898      837.296
 -168.755    237.898     350.367       -18.551     -56.3265     238.347
 -790.138   -218.005   -1146.08     …  -32.6276    146.184    -1798.38
 -137.949     22.5204    -17.6735      -19.9898     -6.73469    -62.9694
   68.9745   -11.2602      8.83673       9.9949      3.36735     31.4847
 -261.163     95.7347     61.7551        3.36735   273.551     -159.898
  887.423    207.719    1054.51         31.4847   -159.898     1897.95

In [362]:
# MME

In [363]:
LHS = [X'*Dinv*X X'*Dinv*genoRef;
       genoRef'*Dinv*X genoRef'*Dinv*genoRef+Matrix(1.0I,size(genoRef,2),size(genoRef,2))*αM]

11×11 Matrix{Union{Missing, Float64}}:
 1959.0      965.643   -157.643   …  139.929      47.1429     440.786
  965.643   1314.47    -306.301       68.9745   -261.163      887.423
 -157.643   -306.301    429.328      -11.2602     95.7347     207.719
  123.714     47.1837   323.673        8.83673    61.7551    1054.51
 -213.286     50.8265   180.031      -15.2347     20.898      837.296
 -259.714   -168.755    237.898   …  -18.551     -56.3265     238.347
 -456.786   -790.138   -218.005      -32.6276    146.184    -1798.38
 -279.857   -137.949     22.5204     -19.9898     -6.73469    -62.9694
  139.929     68.9745   -11.2602      34.5934      3.36735     31.4847
   47.1429  -261.163     95.7347       3.36735   298.149     -159.898
  440.786    887.423    207.719   …   31.4847   -159.898     1922.55

In [364]:
RHS = [X'*Dinv*y;
       genoRef'*Dinv*y]

11-element Vector{Union{Missing, Float64}}:
 21754.900000000005
 10213.407142857142
 -2488.5071428571437
   805.1857142857135
 -3646.7142857142862
 -2405.6857142857143
 -2803.064285714283
 -3107.842857142857
  1553.9214285714277
  -166.54285714285572
  2978.5642857142857

In [365]:
solD = LHS\RHS

11-element Vector{Float64}:
 11.876337517910018
 -0.6328814977050494
 -3.0407446090004693
  3.069271181204217
 -1.2673936750285497
  2.5998994809834963
  4.446558657297773
  2.4493429072130197e-14
 -3.0812142047720496e-14
 -3.2404452130974613
  1.8830699427401008

In [366]:
# Compute DGV for the reference population 
uHatRefD = genoRef*solD[2:end]

8-element Vector{Float64}:
 -2.6508236451364633
  1.3071577755082628
  0.6104839057558298
  1.0072540329380626
 -5.693198933755382
 -4.358355778878116
  0.5014715662915197
 -5.718438024978174

In [367]:
# Compute DGV for the selection candidates
uHatCandD = genoCand*solD[2:end]

6-element Vector{Float64}:
 -0.006142494014309108
  6.51299029226572
 -3.835368082238073
  2.700943812291447
  3.2725450791682595
  6.349480494781422